# 15. Individual backward feature selection

Start from Notebook 14's full pool: 17 numeric/flag features plus month and weekday encoding (34 columns), region FE additional. Every numeric feature can be removed individually. Month is one categorical unit; weekday and ZIP FE remain forced controls. Fully redundant representations and static FE-absorbed columns remain excluded for the reasons documented in Notebook 14.

Selection uses January-June 2021 only. At each round, refit every single-deletion candidate on the same 585-region cohort and forecast dates; select the lowest mean MAE across horizons 1/2/3. Accept a deletion only when MAE improves at least 0.1% relative to the current model. Refit and repeat until no deletion meets that rule. This declared heuristic selects predictive performance, not coefficient significance. RMSE and each horizon are also recorded.

Freeze the resulting feature list before checking July-December 2021. Later observed outcomes can enter subsequent rolling parameter refits; feature membership is unchanged. These periods have appeared in earlier validation, so this is exploratory temporal stability assessment, not a historically untouched holdout. No 2022-2023 test evaluation.

Save paired date-level HAC loss tests, 14-day block CIs, and Holm adjustment over all examined deletion contrasts. Adaptive selection still invalidates a confirmatory interpretation: neither adjusted p-values nor pointwise CIs account fully for the search. No automatic causal or final-test claims.


In [ ]:
"""Validation-only feature ablations and exploratory inference for NJ onset counts.

Requires numpy/pandas. Optional matplotlib creates report figures. No GMM tests.
"""
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import platform
import numpy as np
import pandas as pd

REPO=Path(r'C:\Users\41240\OneDrive\文档\GitHub\practicum-project\Weikun')
SOURCE=REPO/'nj_daily_feature_outputs'/'nj_daily_features.csv.gz'

BASE=['cases_today','cases_lag_1','cases_lag_7']
HISTORY=BASE+['cases_mean_7d']
EXTENDED=HISTORY+['cases_lag_14','case_std_7d']
ACC=EXTENDED+['case_acceleration']
DYNAMICS=ACC+['case_jerk']
SPECIFICATIONS={
    'B0_basic':BASE,
    'H0_weekly_mean':HISTORY,
    'H1_history_extended':EXTENDED,
    'D1_acceleration':ACC,
    'D2_jerk':DYNAMICS,
    'P0_basic_pressure7':BASE+['nj_covid_pressure_7d'],
    'P1_rich_pressure7':DYNAMICS+['nj_covid_pressure_7d'],
    'P2_rich_pressure14':DYNAMICS+['nj_covid_pressure_14d'],
}
COMPARISONS=[
    ('B0_basic','H0_weekly_mean','weekly_mean'),
    ('H0_weekly_mean','H1_history_extended','lag14_and_volatility'),
    ('H1_history_extended','D1_acceleration','acceleration'),
    ('D1_acceleration','D2_jerk','jerk'),
    ('B0_basic','P0_basic_pressure7','pressure7_basic'),
    ('D2_jerk','P1_rich_pressure7','pressure7_rich'),
    ('D2_jerk','P2_rich_pressure14','pressure14_rich'),
    ('P1_rich_pressure7','P2_rich_pressure14','pressure14_vs_pressure7'),
]


def two_way_meat(scores,n_parameters):
    """CR1 two-way cluster sandwich meat: region + date - region/date cell."""
    N,T,p=scores.shape
    n=N*T
    if min(N,T)<=1 or n<=n_parameters:
        return np.full((p,p),np.nan)
    base=(n-1)/(n-n_parameters)
    by_region=scores.sum(axis=1)
    by_date=scores.sum(axis=0)
    cells=scores.reshape(-1,p)
    return base*(N/(N-1)*(by_region.T@by_region)+T/(T-1)*(by_date.T@by_date)-n/(n-1)*(cells.T@cells))


def fit_fe(x,y):
    N,T,p=x.shape
    assert np.isfinite(x).all() and np.isfinite(y).all()
    xm=x.mean(axis=1); ym=y.mean(axis=1)
    xc=x-xm[:,None,:]; yc=y-ym[:,None]
    scale=np.sqrt(np.mean(xc*xc,axis=(0,1)))
    scale=np.where(scale>1e-12,scale,1)
    z=xc/scale
    zf=z.reshape(-1,p)
    b,_,rank,sv=np.linalg.lstsq(zf,yc.reshape(-1),rcond=None)
    beta=b/scale; alpha=ym-xm@beta
    residual=yc-np.einsum('ntp,p->nt',z,b)
    if rank==p:
        bread=np.linalg.pinv(zf.T@zf,rcond=1e-12)
        covariance=bread@two_way_meat(z*residual[:,:,None],N+p)@bread
        variance=np.diag(covariance)/(scale*scale)
        se=np.full(p,np.nan)
        positive=np.isfinite(variance)&(variance>0)
        se[positive]=np.sqrt(variance[positive])
    else:
        se=np.full(p,np.nan)
    pvalue=np.array([math.erfc(abs(v/s)/math.sqrt(2)) if np.isfinite(s) and s>0 else np.nan for v,s in zip(beta,se)])
    condition=float(sv[0]/sv[-1]) if len(sv) and sv[-1]>0 else np.inf
    return beta,alpha,se,pvalue,int(rank),condition


def hac_mean_test(d,bandwidth=14):
    """Two-sided asymptotic test of zero mean paired DATE-level loss difference."""
    d=np.asarray(d,float)
    assert np.isfinite(d).all()
    T=len(d); L=min(int(bandwidth),T-1)
    mean=float(d.mean()); c=d-mean
    longvar=float(c@c/T)
    for lag in range(1,L+1):
        longvar+=2*(1-lag/(L+1))*float(c[lag:]@c[:-lag]/T)
    variance=longvar/T*T/(T-1)
    if variance<=0 or not np.isfinite(variance):
        return mean,np.nan,np.nan
    se=math.sqrt(variance)
    return mean,se,math.erfc(abs(mean/se)/math.sqrt(2))


def holm_adjust(p):
    p=np.asarray(p,float); adjusted=np.full(p.shape,np.nan)
    good=np.flatnonzero(np.isfinite(p)); ordered=good[np.argsort(p[good])]
    if len(ordered):
        values=np.maximum.accumulate((len(ordered)-np.arange(len(ordered)))*p[ordered])
        adjusted[ordered]=np.minimum(values,1)
    return adjusted


def block_indices(T,block_length,repetitions,seed):
    """Noncircular moving-block resampling; one date index shared by all regions/horizons."""
    assert 1<=block_length<=T
    rng=np.random.default_rng(seed)
    starts=rng.integers(0,T-block_length+1,size=(repetitions,math.ceil(T/block_length)))
    return (starts[:,:,None]+np.arange(block_length)[None,None,:]).reshape(repetitions,-1)[:,:T]


def residual_correlation(errors,lag):
    a=errors[:,lag:]; b=errors[:,:-lag]
    a=a-a.mean(axis=1,keepdims=True); b=b-b.mean(axis=1,keepdims=True)
    den=np.sqrt(np.sum(a*a,axis=1)*np.sum(b*b,axis=1))
    value=np.full(len(a),np.nan)
    good=den>0; value[good]=np.sum(a[good]*b[good],axis=1)/den[good]
    return value


def self_checks():
    # FWL estimates and cluster covariance versus an explicit dummy design.
    rng=np.random.default_rng(729)
    N,T,p=5,12,2
    x=rng.normal(size=(N,T,p)); y=x@np.array([.5,-.25])+np.arange(N)[:,None]+rng.normal(size=(N,T))
    beta,alpha,se,pv,rank,_=fit_fe(x,y)
    design=np.column_stack([np.repeat(np.eye(N),T,axis=0),x.reshape(-1,p)])
    full=np.linalg.lstsq(design,y.reshape(-1),rcond=None)[0]
    assert np.allclose(full[:N],alpha) and np.allclose(full[N:],beta)
    e=y.reshape(-1)-design@full
    inv=np.linalg.pinv(design.T@design)
    covariance=inv@two_way_meat((design*e[:,None]).reshape(N,T,N+p),N+p)@inv
    positive=np.diag(covariance)[N:]>0
    assert np.allclose(se[positive]**2,np.diag(covariance)[N:][positive])
    assert np.allclose(holm_adjust([.01,.04,.03]),[.03,.06,.06])
    independent=rng.normal(size=100)
    _,s,_=hac_mean_test(independent,0)
    assert np.isclose(s,np.std(independent,ddof=1)/10)



GROUPS = {
    'core_history': ['cases_today','cases_lag_1','cases_lag_7','cases_lag_14'],
    'rolling_level': ['cases_mean_7d','cases_mean_14d'],
    'volatility': ['case_std_7d','case_std_14d'],
    'dynamics': ['case_acceleration','case_jerk'],
    'local_rates': ['case_rate_today_per_100k','case_rate_7d_per_100k','case_rate_14d_per_100k'],
    'sparse_history': ['days_since_last_observed_case_censored','no_previous_observed_case'],
    'pressure': ['nj_covid_pressure_7d','nj_covid_pressure_14d'],
    'month': [f'target_month_{k}' for k in range(2,13)],
}
WEEKDAY=[f'target_weekday_{k}' for k in range(2,8)]

def independent_columns(x):
    # Training-only rank filtering. Removes algebraic duplicates and calendar
    # categories absent from the current window; not predictive feature selection.
    z=x-x.mean(axis=1,keepdims=True)
    norms=np.sqrt(np.mean(z*z,axis=(0,1)))
    selected=[]; q=[]
    for j in range(z.shape[2]):
        if norms[j]<1e-10: continue
        v=(z[:,:,j]/norms[j]).reshape(-1).copy()
        for _ in range(2):
            for u in q:v-=u*(u@v)
        length=np.linalg.norm(v)
        if length>1e-7*np.sqrt(v.size):
            selected.append(j);q.append(v/length)
    return selected


def run_individual_selection(source,output,window=30,min_relative_improvement=0.001,bootstrap_repetitions=2000):
    """Greedy backward selection on early 2021; later 2021 checked after freezing."""
    self_checks()
    source=Path(source)
    folder=Path(output)/('individual_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    folder.mkdir(parents=True)
    numeric=sum([v for k,v in GROUPS.items() if k!='month'],[])
    full=numeric+GROUPS['month']+WEEKDAY
    cols=sorted(set(numeric)-set(GROUPS['sparse_history']))
    p=pd.read_csv(source,usecols=cols+['zip','date','population','days_since_last_observed_case','pressure_definition'],dtype={'zip':'string'},parse_dates=['date'])
    assert p.pressure_definition.eq('valid_population_nj_origins_renormalized_v2').all()
    assert not p.duplicated(['zip','date']).any()
    ids=np.array(sorted(p.zip.unique()));dates=pd.date_range(p.date.min(),p.date.max())
    def matrix(c):return p.pivot(index='zip',columns='date',values=c).reindex(index=ids,columns=dates).to_numpy(float)
    population=matrix('population')[:,0]
    eligible=(population>0)&np.isfinite(population)&np.isfinite(matrix('nj_covid_pressure_14d')[:,13])
    matrices={c:matrix(c)[eligible] for c in cols}
    gap=matrix('days_since_last_observed_case')[eligible]
    matrices['no_previous_observed_case']=(~np.isfinite(gap)).astype(float)
    matrices['days_since_last_observed_case_censored']=np.where(np.isfinite(gap),gap,np.arange(len(dates))[None,:]+1)
    ids=ids[eligible];n=len(ids);counts=matrices['cases_today']
    origins=np.flatnonzero((dates>=pd.Timestamp('2021-01-01'))&((dates+pd.Timedelta(days=3))<=pd.Timestamp('2021-12-31')))
    v=len(origins);actual=np.stack([counts[:,origins+h] for h in (1,2,3)])
    selection_mask=(dates[origins]+pd.Timedelta(days=3))<=pd.Timestamp('2021-06-30')
    assessment_mask=dates[origins]>=pd.Timestamp('2021-07-01')
    assert not (selection_mask&assessment_mask).any()
    # Cached sufficient statistics let every candidate refit the actual OLS
    # on its own feature subset without repeating full raw training-data scans.
    cache=[];timing=[]
    for h in (1,2,3):
        target=dates+pd.Timedelta(days=h)
        for k in range(2,8):matrices[f'target_weekday_{k}']=np.broadcast_to((target.dayofweek==k-1).astype(float),(n,len(dates)))
        for k in range(2,13):matrices[f'target_month_{k}']=np.broadcast_to((target.month==k).astype(float),(n,len(dates)))
        x=np.stack([matrices[c] for c in full],axis=2)
        blocks=[]
        for first in range(0,v,7):
            block=origins[first:first+7];fit=int(block[0]);last=fit-h;train=np.arange(last-window+1,last+1)
            assert train[0]>=14 and train[-1]+h<=fit
            xt=x[:,train];yt=counts[:,train+h]
            assert np.isfinite(xt).all() and np.isfinite(x[:,block]).all()
            xm=xt.mean(axis=1);ym=yt.mean(axis=1);xc=xt-xm[:,None];yc=yt-ym[:,None]
            scale=np.sqrt((xc*xc).mean(axis=(0,1)));scale=np.where(scale>1e-10,scale,1)
            z=xc/scale;zf=z.reshape(-1,len(full))
            gram=zf.T@zf;rhs=zf.T@yc.reshape(-1)
            blocks.append((first,len(block),gram,rhs,scale,xm,ym,x[:,block]))
            timing.append({'horizon':h,'fit_date':dates[fit],'training_origin_start':dates[train[0]],'training_origin_end':dates[train[-1]],'latest_training_target':dates[train[-1]+h]})
        cache.append(blocks)
    np.seterr(invalid='raise')
    def predict(features,return_rank_log=False):
        ix=[full.index(f) for f in features]
        predictions=np.empty((3,n,v));rank_log=[]
        for hh,blocks in enumerate(cache):
            for first,length,g,rhs,scale,xm,ym,xp in blocks:
                # Ordered Gram-Schmidt in training Gram space removes only
                # nonidentified columns. Standardization is training-only.
                selected=[]
                for j in ix:
                    if g[j,j]<1e-8:continue
                    if not selected:residual=g[j,j]
                    else:
                        gg=g[np.ix_(selected,selected)];cross=g[selected,j]
                        residual=g[j,j]-cross@np.linalg.solve(gg,cross)
                    if residual>1e-8*max(g[j,j],1):selected.append(j)
                if selected:
                    b=np.linalg.solve(g[np.ix_(selected,selected)],rhs[selected])/scale[selected]
                    a=ym-xm[:,selected]@b
                    predictions[hh,:,first:first+length]=np.einsum('ntp,p->nt',xp[:,:,selected],b)+a[:,None]
                else:predictions[hh,:,first:first+length]=ym[:,None]
                if return_rank_log:rank_log.append({'horizon':hh+1,'forecast_origin':dates[origins[first]],'estimated_slopes':len(selected),'omitted':';'.join(full[j] for j in ix if j not in selected)})
        assert np.isfinite(predictions).all()
        return (predictions,rank_log) if return_rank_log else predictions
    def losses(pred,mask):
        e=np.maximum(pred,0)[:,:,mask]-actual[:,:,mask]
        return np.abs(e).mean(axis=1),(e*e).mean(axis=1),e.mean(axis=(1,2))
    # Numerical regression check: one subset must match direct within OLS.
    hh=0;first,length,g,rhs,scale,xm,ym,xp=cache[hh][0]
    check_features=['cases_today','cases_lag_1','cases_lag_7']+WEEKDAY
    check_ix=[full.index(f) for f in check_features]
    train_end=int(origins[0])-1;tr=np.arange(train_end-window+1,train_end+1)
    check_x=np.stack([matrices[f] if f not in WEEKDAY else np.broadcast_to(((dates+pd.Timedelta(days=1)).dayofweek==int(f.rsplit('_',1)[1])-1).astype(float),(n,len(dates))) for f in check_features],axis=2)
    direct_b,direct_a,*_=fit_fe(check_x[:,tr],counts[:,tr+1])
    direct=np.einsum('ntp,p->nt',check_x[:,origins[:length]],direct_b)+direct_a[:,None]
    assert np.allclose(predict(check_features)[0,:,:length],direct,atol=1e-7,rtol=1e-7)
    candidates={f:[f] for f in numeric};candidates['target_month']=GROUPS['month']
    protocol={'source':str(source.resolve()),'source_sha256':hashlib.sha256(source.read_bytes()).hexdigest(),'regions':n,'window':window,
        'horizons':[1,2,3],'retrain_every_days':7,'selection_period':'2021-01-01 through 2021-06-30; all three targets inside period',
        'assessment_period':'2021-07-01 through 2021-12-31; frozen features; parameters still refit using available past outcomes',
        'minimum_relative_MAE_improvement':min_relative_improvement,'stopping_rule':'Stop unless best single deletion lowers mean early-period validation MAE by at least relative threshold. Deterministic feature-name tie break.',
        'forced_features':WEEKDAY,'region_FE':True,'initial_features':full,'candidate_units':candidates,
        'statistical_status':'Exploratory. Earlier 2021 results already inspected. Adaptive p-values are not post-selection-valid. Assessment never enters this algorithm but is not a historically untouched holdout.',
        'test_2022_2023_evaluated':False,'decision_metric':'Mean horizon MAE, not coefficient or loss p-values. RMSE recorded, not optimized.'}
    (folder/'protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
    pd.DataFrame({'zip':ids}).to_csv(folder/'common_regions.csv',index=False)
    current=full.copy();current_pred=predict(current);full_pred=current_pred.copy()
    steps=[];trials=[];fit_count=1;round_i=0
    while candidates:
        round_i+=1
        ae,mse,_=losses(current_pred,selection_mask);current_mae=float(ae.mean());current_rmse=float(np.sqrt(mse.mean(axis=1)).mean())
        round_trials=[];best=None
        for label,remove in candidates.items():
            features=[f for f in current if f not in remove]
            pred=predict(features);fit_count+=1
            ca,cm,_=losses(pred,selection_mask);mae=float(ca.mean());rmse=float(np.sqrt(cm.mean(axis=1)).mean())
            difference=(ae-ca).mean(axis=0)
            delta,se,pv=hac_mean_test(difference,14)
            index=block_indices(len(difference),14,bootstrap_repetitions,20261028+round_i)
            low,high=np.quantile(difference[index].mean(axis=1),[.025,.975])
            row={'round':round_i,'removed_candidate':label,'current_MAE':current_mae,'candidate_MAE':mae,'current_RMSE':current_rmse,'candidate_RMSE':rmse,
                'delta_MAE_current_minus_candidate':delta,'relative_MAE_improvement':delta/current_mae,'ci95_low':low,'ci95_high':high,'p_HAC_raw_exploratory':pv,
                'candidate_feature_columns':len(features),'chosen':False}
            for h in range(3):row[f'MAE_h{h+1}']=float(ca[h].mean())
            round_trials.append(row)
            if best is None or (mae,label)<(best[0],best[1]):best=(mae,label,features,pred,row)
        accepted=(current_mae-best[0])/current_mae>=min_relative_improvement
        if accepted:best[4]['chosen']=True
        for row in round_trials:trials.append(row)
        steps.append({'round':round_i,'removed_feature':best[1] if accepted else 'STOP','best_attempt':best[1],
            'MAE_before':current_mae,'MAE_after':best[0] if accepted else current_mae,'accepted':accepted,
            'remaining_features':';'.join(best[2] if accepted else current)})
        pd.DataFrame(steps).to_csv(folder/'selection_path.csv',index=False)
        pd.DataFrame(trials).to_csv(folder/'candidate_trials.csv',index=False)
        print(f'Round {round_i}: best deletion={best[1]}, MAE {current_mae:.6f} -> {best[0]:.6f}, accepted={accepted}',flush=True)
        if not accepted:break
        current=best[2];current_pred=best[3];del candidates[best[1]]
    trials=pd.DataFrame(trials);trials['p_Holm_all_examined_exploratory']=holm_adjust(trials.p_HAC_raw_exploratory)
    trials.to_csv(folder/'candidate_trials.csv',index=False)
    selected_pred,rank_log=predict(current,True)
    # Freeze the list before opening later-period losses.
    selected={'selected_columns':current,'removed_columns':[f for f in full if f not in current],
        'feature_columns':len(current),'forced_weekday_columns':6,'selection_metric':'early-2021 mean MAE','final_test_claim':False}
    (folder/'selected_features.json').write_text(json.dumps(selected,indent=2),encoding='utf-8')
    results=[];final_tests=[];daily=[]
    for period,mask in [('selection',selection_mask),('later_validation',assessment_mask)]:
        for model,pred in [('FULL',full_pred),('SELECTED',selected_pred)]:
            ae,mse,bias=losses(pred,mask)
            for h in range(3):
                results.append({'period':period,'model':model,'horizon':h+1,'regions':n,'forecast_dates':int(mask.sum()),'MAE':float(ae[h].mean()),'RMSE':float(np.sqrt(mse[h].mean())),'bias':float(bias[h])})
                for day,aa,mm in zip(dates[origins[mask]],ae[h],mse[h]):daily.append({'period':period,'model':model,'horizon':h+1,'date':day,'MAE':aa,'MSE':mm})
        f,_m,_b=losses(full_pred,mask);s,_m,_b=losses(selected_pred,mask)
        d=(f-s).mean(axis=0);delta,se,pv=hac_mean_test(d,14)
        draws=block_indices(len(d),14,bootstrap_repetitions,20261029)
        lo,hi=np.quantile(d[draws].mean(axis=1),[.025,.975])
        final_tests.append({'period':period,'delta_MAE_FULL_minus_SELECTED':delta,'ci95_low':lo,'ci95_high':hi,'p_HAC_exploratory':pv})
    pd.DataFrame(results).to_csv(folder/'full_vs_selected_by_horizon.csv',index=False)
    pd.DataFrame(final_tests).to_csv(folder/'full_vs_selected_uncertainty.csv',index=False)
    pd.DataFrame(daily).to_csv(folder/'final_daily_losses.csv',index=False)
    pd.DataFrame(timing).to_csv(folder/'training_timing.csv',index=False)
    pd.DataFrame(rank_log).to_csv(folder/'selected_rank_log.csv',index=False)
    np.savez_compressed(folder/'final_predictions.npz',raw_predictions=np.stack([full_pred,selected_pred]),actual_cases=actual,models=np.array(['FULL','SELECTED']),zip=ids,date=dates[origins].to_numpy(),horizons=np.array([1,2,3]),selection_mask=selection_mask,assessment_mask=assessment_mask)
    report='# Individual backward feature selection\n\n'
    report+=f'Common regions: {n}; window {window} days, weekly refits, horizons 1/2/3. Initial columns {len(full)}, retained {len(current)}, including six weekday controls. ZIP FE are additional. No 2022-2023 test data scored.\n\n'
    report+='Single numeric features were considered for deletion separately. Month dummies remain a single categorical unit. Weekday and region FE are forced. Every accepted removal is followed by a new round of refits. Stop if the best deletion fails to improve early-2021 mean MAE by at least 0.1%. P-values do not choose deletions. This threshold is a declared heuristic, not a statistical guarantee.\n\n'
    report+='The feature list is frozen using January-June 2021 only, then checked July-December 2021. Parameters in that later period may use prior observed outcomes when refitting, as in the rolling forecast design. These periods have been examined in earlier work: neither results nor p-values are confirmatory. Adaptive selection biases inference; Holm adjustment over examined contrasts does not repair that bias. CIs are pointwise date-block intervals from fixed forecasts, not full-pipeline resampling.\n\n'
    report+='## Retained columns\n\n'+''.join('- '+f+'\n' for f in current)
    report+='\n## Removed columns\n\n'+''.join('- '+f+'\n' for f in full if f not in current)
    report+='\nStart with selection_path.csv, selected_features.json and full_vs_selected_by_horizon.csv. candidate_trials.csv records every attempted deletion. Features omitted temporarily for training rank are logged separately; this is not selection. Static FE-absorbed features remain candidates for tree models. Pressure removal, if selected, does not prove it lacks value for other estimators, forecast horizons or warning outputs.\n'
    (folder/'RESULTS.md').write_text(report,encoding='utf-8')
    checks={'self_checks_passed':True,'cached_subset_OLS_matches_direct_within_OLS':True,'all_predictions_finite':True,
        'all_training_targets_at_or_before_fit_date':True,'regions':n,'forecast_dates':v,'evaluated_specifications':fit_count,
        'accepted_removals':sum(s['accepted'] for s in steps),'retained_columns':len(current),'test_evaluated':False}
    (folder/'verification.json').write_text(json.dumps(checks,indent=2),encoding='utf-8')
    print(pd.DataFrame(results).to_string(index=False),flush=True);print('Saved to:',folder.resolve(),flush=True)
    return pd.DataFrame(steps),pd.DataFrame(results),selected,folder


In [ ]:
SOURCE = Path.cwd() / 'nj_daily_feature_outputs' / 'nj_daily_features.csv.gz'
OUTPUT = Path.cwd() / 'individual_feature_selection_outputs'
path, results, selected, RUN_FOLDER = run_individual_selection(SOURCE, OUTPUT)
display(path)
display(results)
print(selected)
print(RUN_FOLDER.resolve())


## Inspect saved results

1. selection_path.csv: accepted deletions, order and stopping step.
2. selected_features.json: retained and removed columns.
3. full_vs_selected_by_horizon.csv: early-selection and later-validation MAE/RMSE/bias.
4. candidate_trials.csv: every attempted deletion, interval and exploratory p-values.
5. full_vs_selected_uncertainty.csv: final paired loss comparisons.
6. protocol.json and verification.json: data hash, decisions and timing checks.

Check the frozen list using 60-day windows before finalizing. This list is specific to FE OLS and does not dictate tree-model inputs. Residual Moran's I requires a separate real geographic weights definition.
